# smoking-health-eda

원본 분석 코드의 공개용 복사본입니다. 원본 셀 번호는 메타데이터에 기록했습니다. 교육용 문제 설명·개별 기록 출력·계정/개인 경로를 제거했습니다. 데이터는 별도 승인 후 확보해야 합니다. 먼저 프로젝트 README를 읽으세요.

In [ ]:
from pathlib import Path
import os
import platform
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if PROJECT_ROOT.name != "smoking-health-eda":
    raise RuntimeError("작업 디렉터리를 projects/smoking-health-eda 또는 그 notebooks 폴더로 설정하세요.")
DATA_DIR = Path(os.environ.get("HEALTHCARE_DATA_DIR", str(PROJECT_ROOT / "data")))
ALLOW_TRAINING = False  # 학습 셀 실행을 원할 때만 직접 True로 변경


## 필요한 라이브러리들 불러오기

원본 코드 셀 4. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# 필요한 라이브러리들 불러오기
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# 한글 폰트 설정 (그래프에서 한글 깨짐 방지)
plt.rc('font', family=('Malgun Gothic' if platform.system() == 'Windows' else 'DejaVu Sans'))
plt.rcParams['axes.unicode_minus'] = False # 마이너스 기호 깨짐 방지

print("라이브러리 로딩 완료!")


## 데이터 로딩

원본 코드 셀 6. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# 데이터 로딩
health_data = pd.read_csv(DATA_DIR / 'smoking_health_data.csv')


## 데이터 크기 출력

원본 코드 셀 7. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# 데이터 크기 출력
print(f"데이터의 크기: {health_data.shape}")


## health_data 각 컬럼의 데이터 타입, Non-null 값 개수 등을 확인

원본 코드 셀 9. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# health_data 각 컬럼의 데이터 타입, Non-null 값 개수 등을 확인
health_data.info()


## health_data의 숫자형 컬럼의 통계 요약 (평균, 표준편차, 최솟값, 최댓값 등)을 확인

원본 코드 셀 10. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# health_data의 숫자형 컬럼의 통계 요약 (평균, 표준편차, 최솟값, 최댓값 등)을 확인
health_data.describe()


## health_data의 상위 5행을 확인

원본 코드 셀 11. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# health_data의 상위 5행을 확인
health_data.head()


## health_data의 하위 5행을 확인

원본 코드 셀 12. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# health_data의 하위 5행을 확인
health_data.tail()


## BMI를 기준으로 체중 상태 구분 컬럼 함수

원본 코드 셀 14. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# BMI를 기준으로 체중 상태 구분 컬럼 함수
def categorize_bmi(bmi):
    if bmi < 18.5:
        return '저체중'
    elif bmi < 23:
        return '정상'
    elif bmi < 25:
        return '비만전단계'
    elif bmi < 30:
        return '1단계 비만'
    elif bmi < 35:
        return '2단계 비만'
    else:
        return '3단계 비만'


## BMI를 기준으로 체중 상태 구분

원본 코드 셀 15. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# BMI를 기준으로 체중 상태 구분
health_data['BMI 구간'] = health_data['BMI'].apply(categorize_bmi)
health_data[['BMI', 'BMI 구간']].head()


## BMI 구간 순서 지정

원본 코드 셀 16. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# BMI 구간 순서 지정
bmi_order = [
    '저체중',
    '정상',
    '비만전단계',
    '1단계 비만',
    '2단계 비만',
    '3단계 비만',
]

health_data['BMI 구간'] = pd.Categorical(
    health_data['BMI 구간'],
    categories=bmi_order,
    ordered=True
)


## 나이를 기준으로 나이대 구분 컬럼 함수

원본 코드 셀 18. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# 나이를 기준으로 나이대 구분 컬럼 함수
def categorize_age(age):
    if 20 <= age < 30:
        return '20대'
    elif 30 <= age < 40:
        return '30대'
    elif 40 <= age < 50:
        return '40대'
    elif 50 <= age < 60:
        return '50대'
    elif 60 <= age < 70:
        return '60대'
    elif 70 <= age < 80:
        return '70대'
    else:
        return '70대 초과'


## 나이를 기준으로 나이대 구분

원본 코드 셀 19. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# 나이를 기준으로 나이대 구분
health_data['나이대'] = health_data['나이'].apply(categorize_age)
health_data[['나이', '나이대']].head()


## 나이대 순서 지정

원본 코드 셀 20. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# 나이대 순서 지정
age_order = [
    '20대',
    '30대',
    '40대',
    '50대',
    '60대',
    '70대',
    '70대 초과'
]

health_data['나이대'] = pd.Categorical(
    health_data['나이대'],
    categories=age_order,
    ordered=True
)


## BMI 기준으로 중성지방 구간 나누는 함수

원본 코드 셀 21. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# BMI 기준으로 중성지방 구간 나누는 함수
def tg_categorize_bmi(tg):
    if tg <= 149:
        return '정상'
    elif tg <= 199:
        return '경계'
    elif tg <= 400:
        return '위험'


## BMI 기준으로 중성지방 구간 나누기

원본 코드 셀 22. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# BMI 기준으로 중성지방 구간 나누기
health_data['중성 지방 구간'] = health_data['중성 지방'].apply(tg_categorize_bmi)
health_data[['중성 지방', '중성 지방 구간']].head(20)


## 중성지방 구간 순서 지정

원본 코드 셀 23. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# 중성지방 구간 순서 지정
tg_order = [
    '정상',
    '경계',
    '위험'
]

health_data['중성 지방 구간'] = pd.Categorical(
    health_data['중성 지방 구간'],
    categories=tg_order,
    ordered=True
)


## 컬럼별 결측치 개수

원본 코드 셀 25. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# 컬럼별 결측치 개수
missing_count = health_data.isnull().sum()  # Series반환 (True=1, False=0으로 계산)
print("컬럼별 결측치 개수:")
print(missing_count)

# 전체 결측치 개수
total_missing = missing_count.sum()   # 모든 결측치 데이터 합산
print(f"전체 결측치: {total_missing}개")

# 결측치 비율 계산
missing_ratio = (missing_count / len(health_data)) * 100
print("\n컬럼별 결측치 비율:")
for col, ratio in missing_ratio.items():
    print(f"{col}: {ratio:.1f}%")


## 혈압 → 나이대별 평균

원본 코드 셀 27. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# 혈압 → 나이대별 평균
health_data['혈압'] = health_data.groupby('나이대')['혈압'].transform(lambda x: x.fillna(x.mean()))

# 시력 → 나이대별 중앙값
health_data['시력'] = health_data.groupby('나이대')['시력'].transform(lambda x: x.fillna(x.median()))

# 중성 지방 → BMI 구간별 평균
health_data['중성 지방'] = health_data.groupby('BMI 구간')['중성 지방'].transform(lambda x: x.fillna(x.mean()))

# 중성지방 구간 갱신 (결측치 채워진 중성지방으로 재분류) - '중성지방'은 비어있던 곳은 '중성지방 구간'도 같이 비어 있을 수 밖에 없음
health_data['중성 지방 구간'] = health_data['중성 지방'].apply(tg_categorize_bmi)

# 공복 혈당 → 갱신된 중성지방 구간별 평균
health_data['공복 혈당'] = health_data.groupby('중성 지방 구간')['공복 혈당'].transform(lambda x: x.fillna(x.mean()))

# label -> 흡연 여부
health_data = health_data.rename(columns={'label': '흡연 여부'})
health_data['흡연 여부'] = pd.Categorical(health_data['흡연 여부'], categories=[0, 1], ordered=True)

# 결측치 처리 결과 확인
print("✅ 전처리 및 결측치 대체 완료!")
print(health_data.isnull().sum())
display(health_data.head())


## ==========================================

원본 코드 셀 29. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# ==========================================
# label별 분포 비교 (흡연 vs 비흡연)- Boxplot
# ==========================================

plot_data = health_data.copy()
plot_data['흡연 여부'] = plot_data['흡연 여부'].map({0: '비흡연', 1: '흡연'})

# 숫자형 변수만 선택
numeric_cols = (
    health_data
    .select_dtypes(include=['number'])
    .columns
    .tolist()
)

# label은 범주형 변수이므로 제외
if '흡연 여부' in numeric_cols:
    numeric_cols.remove('흡연 여부')

# 충치, 요 단백도 범주형 성격이므로 제외
for col in ['충치', '요 단백']:
    if col in numeric_cols:
        numeric_cols.remove(col)

n_cols = 2
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(14, n_rows * 3.5)
)

axes = axes.flatten()

for col, ax in zip(numeric_cols, axes):

    sns.boxplot(
        data=health_data,
        x='흡연 여부',
        y=col,
        order=[0, 1],
        ax=ax
    )

    ax.set_title(f'흡연 여부별 {col} 분포 및 이상치 탐지')
    ax.set_xlabel('흡연 여부')
    ax.set_ylabel(col)

# 남는 빈 그래프 숨기기
for ax in axes[len(numeric_cols):]:
    ax.axis('off')


plt.tight_layout()
plt.show()


## ==========================================

원본 코드 셀 30. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# ==========================================
# label별 분포 비교 (흡연 vs 비흡연)- Histplot
# ==========================================

plot_data = health_data.copy()
plot_data['흡연 여부'] = plot_data['흡연 여부'].map({0: '비흡연', 1: '흡연'})

n_cols = 2
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(14, n_rows * 3.5)
)

axes = axes.flatten()

for col, ax in zip(numeric_cols, axes):

    sns.histplot(
        data=plot_data,
        x=col,
        hue='흡연 여부',
        hue_order=['비흡연', '흡연'],
        stat='density',
        common_norm=False,
        bins=25,
        element='step',
        fill=True,
        alpha=0.20,
        kde=True,
        line_kws={'linewidth': 1.8},
        ax=ax
    )

    ax.set_title(f'흡연 여부별 {col} 분포 비교')
    ax.set_xlabel(col)

# 남는 빈 그래프 숨기기
for ax in axes[len(numeric_cols):]:
    ax.axis('off')

plt.tight_layout()
plt.show()


## 가설 1.

원본 코드 셀 33. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# 가설 1.
# H₀ (귀무가설): 중성 지방- 흡연자와 비흡연자의 평균 중성 지방 수치는 같다.
# H₁ (대립가설): 중성 지방- 흡연자와 비흡연자의 평균 중성 지방 수치는 다르다.

# 가설 2.
# H₀ (귀무가설): 헤모글로빈- 흡연자와 비흡연자의 평균 헤모글로빈 수치는 같다.
# H₁ (대립가설): 헤모글로빈- 흡연자와 비흡연자의 평균 헤모글로빈 수치는 다르다.

# 가설 3.
# H₀ (귀무가설): 고밀도지단백- 흡연자와 비흡연자의 평균 고밀도지단백 수치는 같다.
# H₁ (대립가설): 고밀도지단백- 흡연자와 비흡연자의 평균 고밀도지단백 수치는 다르다.


## ==========================================

원본 코드 셀 35. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# ==========================================
# 가설 1 - 중성 지방 단변량 분석
# ==========================================

col = '중성 지방'

# 1. 기초 통계량 확인
display(health_data[[col]].describe().T)

# 중앙값 확인
print(f'{col} 중앙값:', health_data[col].median())

# 결측치 확인
print(f'{col} 결측치 수:', health_data[col].isnull().sum())


# ==========================================
# 2. IQR을 이용한 이상치 계산
# ==========================================

# 제1사분위수(Q1)와 제3사분위수(Q3)
Q1 = health_data[col].quantile(0.25)
Q3 = health_data[col].quantile(0.75)

# IQR = Q3 - Q1
IQR = Q3 - Q1

# 이상치 판단 기준
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# 이상치 데이터 추출
outliers = health_data[
    (health_data[col] < lower_bound) |
    (health_data[col] > upper_bound)
]

# 결과 출력
print(f'Q1: {Q1:.2f}')
print(f'Q3: {Q3:.2f}')
print(f'IQR: {IQR:.2f}')
print(f'이상치 하한값: {lower_bound:.2f}')
print(f'이상치 상한값: {upper_bound:.2f}')
print(f'이상치 개수: {len(outliers)}')
print(f'이상치 비율: {len(outliers) / len(health_data) * 100:.2f}%')


# ==========================================
# 3. Histogram
# ==========================================

plt.figure(figsize=(8, 5))

sns.histplot(
    data=health_data,
    x=col,
    kde=True
)

plt.title(f'{col} 분포')
plt.xlabel(col)
plt.ylabel('빈도')

plt.show()


# ==========================================
# 4. Boxplot
# ==========================================

plt.figure(figsize=(8, 4))

sns.boxplot(
    data=health_data,
    x=col
)

plt.title(f'{col} Boxplot')
plt.xlabel(col)

plt.show()


## ==========================================

원본 코드 셀 36. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# ==========================================
# 가설 2 - 헤모글로빈 단변량 분석
# ==========================================

col = '헤모글로빈'

# 1. 기초 통계량
display(health_data[[col]].describe().T)

print(f'{col} 중앙값:', health_data[col].median())
print(f'{col} 결측치 수:', health_data[col].isnull().sum())


# ==========================================
# 2. IQR을 이용한 이상치 계산
# ==========================================

Q1 = health_data[col].quantile(0.25)
Q3 = health_data[col].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = health_data[
    (health_data[col] < lower_bound) |
    (health_data[col] > upper_bound)
]

print(f'Q1: {Q1:.2f}')
print(f'Q3: {Q3:.2f}')
print(f'IQR: {IQR:.2f}')
print(f'이상치 하한값: {lower_bound:.2f}')
print(f'이상치 상한값: {upper_bound:.2f}')
print(f'이상치 개수: {len(outliers)}')
print(f'이상치 비율: {len(outliers) / len(health_data) * 100:.2f}%')


# ==========================================
# 3. Histogram
# ==========================================

plt.figure(figsize=(8, 5))

sns.histplot(
    data=health_data,
    x=col,
    kde=True
)

plt.title(f'{col} 분포')
plt.xlabel(col)
plt.ylabel('빈도')

plt.show()


# ==========================================
# 4. Boxplot
# ==========================================

plt.figure(figsize=(8, 4))

sns.boxplot(
    data=health_data,
    x=col
)

plt.title(f'{col} Boxplot')
plt.xlabel(col)

plt.show()


## ==========================================

원본 코드 셀 37. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# ==========================================
# 가설 3 - 고밀도지단백 단변량 분석
# ==========================================

col = '고밀도지단백'

# 1. 기초 통계량
display(health_data[[col]].describe().T)

print(f'{col} 중앙값:', health_data[col].median())
print(f'{col} 결측치 수:', health_data[col].isnull().sum())


# ==========================================
# 2. IQR을 이용한 이상치 계산
# ==========================================

Q1 = health_data[col].quantile(0.25)
Q3 = health_data[col].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = health_data[
    (health_data[col] < lower_bound) |
    (health_data[col] > upper_bound)
]

print(f'Q1: {Q1:.2f}')
print(f'Q3: {Q3:.2f}')
print(f'IQR: {IQR:.2f}')
print(f'이상치 하한값: {lower_bound:.2f}')
print(f'이상치 상한값: {upper_bound:.2f}')
print(f'이상치 개수: {len(outliers)}')
print(f'이상치 비율: {len(outliers) / len(health_data) * 100:.2f}%')


# ==========================================
# 3. Histogram
# ==========================================

plt.figure(figsize=(8, 5))

sns.histplot(
    data=health_data,
    x=col,
    kde=True
)

plt.title(f'{col} 분포')
plt.xlabel(col)
plt.ylabel('빈도')

plt.show()


# ==========================================
# 4. Boxplot
# ==========================================

plt.figure(figsize=(8, 4))

sns.boxplot(
    data=health_data,
    x=col
)

plt.title(f'{col} Boxplot')
plt.xlabel(col)

plt.show()


## ==========================================

원본 코드 셀 40. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# ==========================================
# 가설 1 - 중성 지방 단변량 분석
# ==========================================

plot_data = health_data.copy()
plot_data['흡연 여부'] = plot_data['흡연 여부'].map({0: '비흡연', 1: '흡연'})

summary = (
    health_data
    .groupby('흡연 여부')['중성 지방']
    .agg(['count', 'mean', 'median', 'std'])
)

display(summary)


# ==========================================
# 흡연 여부별 Boxplot
# 중앙값 + 범례 표시
# ==========================================
plt.figure(figsize=(7, 5))

ax = sns.boxplot(
    data=plot_data,
    x='흡연 여부',
    y='중성 지방',
    hue='흡연 여부',
    order=['비흡연', '흡연'],
    hue_order=['비흡연', '흡연'],
    dodge=False
)

plt.title('흡연 여부별 중성 지방 분포 및 이상치')
plt.xlabel('흡연 여부')
plt.ylabel('중성 지방')

# x축 이름 변경
plt.xticks(
    [0, 1],
    ['비흡연', '흡연']
)


# ------------------------------------------
# 중앙값 계산 및 표시
# ------------------------------------------

medians = (
    health_data
    .groupby('흡연 여부')['중성 지방']
    .median()
)

for label, median in medians.items():
    ax.text(
        label,
        median,
        f'  중앙값: {median:.1f}',
        va='center',
        ha='left',
        fontweight='bold'
    )

plt.show()


# ==========================================
# 흡연 여부별 Histogram
# 중앙값 + 범례 표시
# ==========================================
plt.figure(figsize=(8, 5))

ax = sns.histplot(
    data=plot_data,
    x='중성 지방',
    hue='흡연 여부',
    hue_order=['비흡연', '흡연'],
    kde=True,
    element='step',
    stat='density',
    common_norm=False
)


# ------------------------------------------
# 중앙값 세로선 표시
# ------------------------------------------

ax.axvline(
    medians[0],
    linestyle='--',
    linewidth=2
)

ax.axvline(
    medians[1],
    linestyle='--',
    linewidth=2
)


# ------------------------------------------
# 중앙값 숫자 표시
# ------------------------------------------

y_max = ax.get_ylim()[1]

ax.text(
    medians[0],
    y_max * 0.95,
    f'비흡연 중앙값\n{medians[0]:.1f}',
    ha='center',
    va='top',
    fontweight='bold'
)

ax.text(
    medians[1],
    y_max * 0.80,
    f'흡연 중앙값\n{medians[1]:.1f}',
    ha='center',
    va='top',
    fontweight='bold'
)


plt.title('흡연 여부별 중성 지방 분포 비교')
plt.xlabel('중성 지방')
plt.ylabel('밀도')

# 기존 seaborn 범례의 제목 설정
ax.legend_.set_title('흡연 여부')

plt.show()



## ==========================================

원본 코드 셀 41. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# ==========================================
# 가설 2 - 흡연 여부와 헤모글로빈 이변량 분석
# ==========================================

plot_data = health_data.copy()
plot_data['흡연 여부'] = plot_data['흡연 여부'].map({0: '비흡연', 1: '흡연'})

summary = (
    health_data
    .groupby('흡연 여부')['헤모글로빈']
    .agg(['count', 'mean', 'median', 'std'])
)

display(summary)


# ==========================================
# 흡연 여부별 Boxplot
# 중앙값 + 범례 표시
# ==========================================
plt.figure(figsize=(7, 5))

ax = sns.boxplot(
    data=plot_data,
    x='흡연 여부',
    y='헤모글로빈',
    hue='흡연 여부',
    hue_order=['비흡연', '흡연'],
    dodge=False
)

plt.title('흡연 여부별 헤모글로빈 분포 및 이상치')
plt.xlabel('흡연 여부')
plt.ylabel('헤모글로빈')

# x축 이름 변경
plt.xticks(
    [0, 1],
    ['비흡연', '흡연']
)


# ------------------------------------------
# 중앙값 계산 및 표시
# ------------------------------------------

medians = (
    health_data
    .groupby('흡연 여부')['헤모글로빈']
    .median()
)

for label, median in medians.items():
    ax.text(
        label,
        median,
        f'  중앙값: {median:.1f}',
        va='center',
        ha='left',
        fontweight='bold'
    )

plt.show()


# ==========================================
# 흡연 여부별 Histogram
# 중앙값 + 범례 표시
# ==========================================

plt.figure(figsize=(8, 5))

ax = sns.histplot(
    data=plot_data,
    x='헤모글로빈',
    hue='흡연 여부',
    hue_order=['비흡연', '흡연'],
    kde=True,
    element='step',
    stat='density',
    common_norm=False
)


# ------------------------------------------
# 중앙값 세로선 표시
# ------------------------------------------

ax.axvline(
    medians[0],
    linestyle='--',
    linewidth=2
)

ax.axvline(
    medians[1],
    linestyle='--',
    linewidth=2
)


# ------------------------------------------
# 중앙값 숫자 표시
# ------------------------------------------

y_max = ax.get_ylim()[1]

ax.text(
    medians[0],
    y_max * 0.95,
    f'비흡연 중앙값\n{medians[0]:.1f}',
    ha='center',
    va='top',
    fontweight='bold'
)

ax.text(
    medians[1],
    y_max * 0.80,
    f'흡연 중앙값\n{medians[1]:.1f}',
    ha='center',
    va='top',
    fontweight='bold'
)


plt.title('흡연 여부별 헤모글로빈 분포 비교')
plt.xlabel('헤모글로빈')
plt.ylabel('밀도')

# 기존 seaborn 범례의 제목 설정
ax.legend_.set_title('흡연 여부')

plt.show()


## ==========================================

원본 코드 셀 42. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# ==========================================
# 가설 3 - 흡연 여부와 고밀도지단백 이변량 분석
# ==========================================

plot_data = health_data.copy()
plot_data['흡연 여부'] = plot_data['흡연 여부'].map({0: '비흡연', 1: '흡연'})

summary = (
    health_data
    .groupby('흡연 여부')['고밀도지단백']
    .agg(['count', 'mean', 'median', 'std'])
)

display(summary)


# ==========================================
# 흡연 여부별 Boxplot
# 중앙값 + 범례 표시
# ==========================================
plt.figure(figsize=(7, 5))

ax = sns.boxplot(
    data=plot_data,
    x='흡연 여부',
    y='고밀도지단백',
    hue='흡연 여부',
    hue_order=['비흡연', '흡연'],
    dodge=False
)

plt.title('흡연 여부별 고밀도지단백 분포 및 이상치')
plt.xlabel('흡연 여부')
plt.ylabel('고밀도지단백')

# x축 이름 변경
plt.xticks(
    [0, 1],
    ['비흡연', '흡연']
)


# ------------------------------------------
# 중앙값 계산 및 표시
# ------------------------------------------

medians = (
    health_data
    .groupby('흡연 여부')['고밀도지단백']
    .median()
)

for label, median in medians.items():
    ax.text(
        label,
        median,
        f'  중앙값: {median:.1f}',
        va='center',
        ha='left',
        fontweight='bold'
    )

plt.show()


# ==========================================
# 흡연 여부별 Histogram
# 중앙값 + 범례 표시
# ==========================================

plt.figure(figsize=(8, 5))

ax = sns.histplot(
    data=plot_data,
    x='고밀도지단백',
    hue='흡연 여부',
    hue_order=['비흡연', '흡연'],
    kde=True,
    element='step',
    stat='density',
    common_norm=False
)


# ------------------------------------------
# 중앙값 세로선 표시
# ------------------------------------------

ax.axvline(
    medians[0],
    linestyle='--',
    linewidth=2
)

ax.axvline(
    medians[1],
    linestyle='--',
    linewidth=2
)


# ------------------------------------------
# 중앙값 숫자 표시
# ------------------------------------------

y_max = ax.get_ylim()[1]

ax.text(
    medians[0],
    y_max * 0.95,
    f'비흡연 중앙값\n{medians[0]:.1f}',
    ha='center',
    va='top',
    fontweight='bold'
)

ax.text(
    medians[1],
    y_max * 0.80,
    f'흡연 중앙값\n{medians[1]:.1f}',
    ha='center',
    va='top',
    fontweight='bold'
)


plt.title('흡연 여부별 고밀도지단백 분포 비교')
plt.xlabel('고밀도지단백')
plt.ylabel('밀도')

# 기존 seaborn 범례의 제목 설정
ax.legend_.set_title('흡연 여부')

plt.show()


## ==========================================

원본 코드 셀 43. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# ==========================================
# 이변량 분석 - t-test
# ==========================================

# 가설 검정 변수
test_cols = [
    '중성 지방',
    '고밀도지단백',
    '헤모글로빈'
]

results = []

for col in test_cols:
    g0 = health_data[health_data['흡연 여부'] == 0][col]
    g1 = health_data[health_data['흡연 여부'] == 1][col]
    t_stat, p_value = stats.ttest_ind(g0, g1, equal_var=False)
    results.append({
        '변수': col,
        '평균(흡연)': g1.mean(),
        '평균(비흡연)': g0.mean(),
        't-statistic': t_stat,
        'p-value': p_value,
        '검정 결과': '귀무가설 기각' if p_value < 0.001 else '귀무가설 기각하지 못함'
    })

result_df = pd.DataFrame(results)
display(result_df)


## ==========================================

원본 코드 셀 44. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
if not ALLOW_TRAINING:
    raise RuntimeError("모델 학습은 ALLOW_TRAINING=True일 때만 실행합니다.")
# ==========================================
# 흡연 여부와 관련된 주요 변수 비교
# ==========================================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import numpy as np
import pandas as pd


# 분석에 사용할 변수
feature_cols = [
    '중성 지방',
    '고밀도지단백',
    '헤모글로빈'
]

X = health_data[feature_cols]
y = health_data['흡연 여부']


# 데이터 크기 확인
print("X shape:", X.shape)
print("y shape:", y.shape)


# ==========================================
# 1. 훈련 데이터 / 평가 데이터 분리
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("훈련 데이터:", X_train.shape)
print("평가 데이터:", X_test.shape)


# ==========================================
# 2. 데이터 표준화
# ==========================================

scaler = StandardScaler()

# 훈련 데이터에서 기준 학습 + 변환
X_train_scaled = scaler.fit_transform(X_train)

# 평가 데이터는 훈련 데이터에서 학습한 기준으로 변환만
X_test_scaled = scaler.transform(X_test)


# ==========================================
# 3. 로지스틱 회귀 모델 학습
# ==========================================

model = LogisticRegression()
model.fit(X_train_scaled, y_train)


# ==========================================
# 4. 예측
# ==========================================

y_pred = model.predict(X_test_scaled)

# label=1(흡연)일 확률
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

print("예측값 (앞 10개):", y_pred[:10])

print("흡연일 확률 (앞 10개):", np.round(y_pred_proba[:10], 2))


# ==========================================
# 5. 모델 성능 평가
# ==========================================

acc = accuracy_score(y_test, y_pred)

print(f"\n정확도(Accuracy): {acc:.3f}")

cm = confusion_matrix(y_test, y_pred)

# 혼동행렬 값 분리
tn, fp, fn, tp = cm.ravel()

print(f"\n실제 비흡연자 {tn + fp}명")
print(f"→ {tn}명 정확")
print(f"→ {fp}명 오분류")
print()

print(f"실제 흡연자 {tp + fn}명")
print(f"→ {tp}명 정확")
print(f"→ {fn}명 오분류")

print("\n분류 리포트")
print(classification_report(y_test, y_pred)
)


# ==========================================
# 6. 주요 영향 요인 확인
# ==========================================

coef_df = pd.DataFrame({
    '변수': feature_cols,
    '계수': model.coef_[0]
})

# 계수의 절댓값으로 상대적인 영향력 비교
coef_df['표준화 계수(절댓값)'] = (
    coef_df['계수'].abs()
)

# 영향력이 큰 순서대로 정렬
coef_df = coef_df.sort_values(
    '표준화 계수(절댓값)',
    ascending=False
).round(3)

display(coef_df)


## ==========================================

원본 코드 셀 45. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# ==========================================
# 가설 검증 시각화
# 흡연자 vs 비흡연자 평균 비교 막대그래프
# ==========================================

import matplotlib.pyplot as plt
from scipy import stats


# 가설 검정 변수
hypothesis_vars = [
    '중성 지방',
    '헤모글로빈',
    '고밀도지단백'
]

titles = {
    '중성 지방': '가설 1. 중성 지방',
    '헤모글로빈': '가설 2. 헤모글로빈',
    '고밀도지단백': '가설 3. 고밀도지단백(HDL)'
}

fig, axes = plt.subplots(
    1, 3,
    figsize=(16, 5)
)


for col, ax in zip(hypothesis_vars, axes):

    # 비흡연자 / 흡연자 데이터
    non_smoker = health_data.loc[
        health_data['흡연 여부'] == 0, col
    ].dropna()

    smoker = health_data.loc[
        health_data['흡연 여부'] == 1, col
    ].dropna()


    # 평균
    non_mean = non_smoker.mean()
    smoker_mean = smoker.mean()

    means = [
        non_mean,
        smoker_mean
    ]


    # Welch t-test
    t_stat, p_value = stats.ttest_ind(
        non_smoker,
        smoker,
        equal_var=False
    )


    # ------------------------------
    # 막대그래프
    # ------------------------------
    bars = ax.bar(
        ['비흡연', '흡연'],
        means
    )


    # ------------------------------
    # 막대 위에 평균값 표시
    # ------------------------------
    for bar, mean in zip(bars, means):

        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f'{mean:.2f}',
            ha='center',
            va='bottom',
            fontsize=12,
            fontweight='bold'
        )


    # ------------------------------
    # 평균 차이 계산
    # ------------------------------
    difference = smoker_mean - non_mean

    if difference > 0:
        diff_text = f'흡연자가 {abs(difference):.2f} 높음 ↑'
    else:
        diff_text = f'흡연자가 {abs(difference):.2f} 낮음 ↓'


    # ------------------------------
    # p-value
    # ------------------------------
    if p_value < 0.001:
        p_text = 'p < 0.001'
    else:
        p_text = f'p = {p_value}'


    # ------------------------------
    # 제목
    # ------------------------------
    ax.set_title(
        f'{titles[col]}\n'
        f'{diff_text}\n'
        f'{p_text}',
        fontsize=12,
        fontweight='bold'
    )

    ax.set_ylabel('평균')
    ax.set_xlabel('흡연 여부')

    ax.grid(
        axis='y',
        alpha=0.2
    )


plt.tight_layout()
plt.show()


## 가설 검정 변수

원본 코드 셀 46. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# 가설 검정 변수
test_cols = [
    '중성 지방',
    '고밀도지단백',
    '헤모글로빈'
]

results = []

for col in test_cols:
    g0 = health_data[health_data['흡연 여부'] == 0][col]
    g1 = health_data[health_data['흡연 여부'] == 1][col]
    t_stat, p_value = stats.ttest_ind(g0, g1, equal_var=False)
    results.append({
        '변수': col,
        '평균(흡연)': g1.mean(),
        '평균(비흡연)': g0.mean(),
        't-statistic': t_stat,
        'p-value': p_value,
        '검정 결과': '귀무가설 기각' if p_value < 0.05 else '귀무가설 기각하지 못함'
    })

result_df = pd.DataFrame(results)
display(result_df)


## ==========================================

원본 코드 셀 49. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# ==========================================
# 범주형 변수를 제외한 수치형 변수 상관관계 분석
# ==========================================

# ==========================================
# 상관계수 계산
# ==========================================

corr_matrix = health_data.corr(numeric_only=True)


# 상삼각 영역 마스킹
mask = np.triu(
    np.ones_like(corr_matrix, dtype=bool)
)


# ==========================================
# 상관계수 히트맵
# ==========================================

plt.figure(figsize=(10, 8))

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    cmap='RdBu_r',
    fmt='.2f',
    cbar_kws={
        'label': '상관계수',
        'shrink': 0.8
    }
)

plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)

plt.title('건강 지표와 흡연여부 간 상관관계')

plt.tight_layout()
plt.show()


# ==========================================
# 흡연 여부와 상관관계가 높은 변수 TOP 10
# ==========================================

smoking_corr = (
    corr_matrix['흡연 여부']
    .drop('흡연 여부')
)


# 절댓값 기준으로 정렬
# 실제 상관계수의 +/- 부호는 유지
smoking_corr = smoking_corr.loc[
    smoking_corr.abs()
    .sort_values(ascending=False)
    .index
]


print("\n📊 흡연 여부와 상관관계가 높은 변수 TOP 10:")
print("=" * 50)

for i, (var, corr) in enumerate(
    smoking_corr.head(10).items(),
    1
):
    direction = (
        '양의 상관'
        if corr > 0
        else '음의 상관'
    )

    print(
        f"{i:2d}. {var:<10}: "
        f"{corr:+.3f} ({direction})"
    )
